In [4]:
import pandas as pd
import numpy as np


In [5]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2023_Ashok_Vihar_Delhi_DPCC_2023.xlsx")

In [6]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,285.0,155.0,178.0,86.0,81.0,87.0,65.0,NaN,127.0,95.0,333.0,382.0
1,2,367.0,191.0,224.0,131.0,66.0,93.0,68.0,NaN,123.0,96.0,358.0,364.0
2,3,393.0,209.0,138.0,146.0,116.0,102.0,98.0,NaN,122.0,99.0,410.0,337.0
3,4,346.0,266.0,112.0,90.0,96.0,150.0,126.0,87.0,122.0,93.0,405.0,331.0
4,5,331.0,258.0,133.0,129.0,176.0,143.0,81.0,70.0,105.0,154.0,NaN,293.0
5,6,393.0,285.0,126.0,63.0,215.0,111.0,51.0,81.0,96.0,192.0,385.0,285.0
6,7,376.0,310.0,147.0,143.0,176.0,NaN,75.0,NaN,76.0,209.0,NaN,351.0
7,8,377.0,140.0,222.0,157.0,126.0,145.0,73.0,NaN,78.0,167.0,NaN,353.0
8,9,424.0,204.0,121.0,245.0,206.0,150.0,63.0,117.0,39.0,174.0,448.0,353.0
9,10,402.0,176.0,170.0,204.0,187.0,139.0,NaN,NaN,40.0,NaN,305.0,318.0


In [7]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [8]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [9]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [10]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,285.0,155.0000,178.000000,86.000000,81.000000,87.0,65.0,97.518519,127.0000,141.090909,333.00000,382.0
1,2,367.0,191.0000,224.000000,131.000000,66.000000,93.0,68.0,97.518519,123.0000,141.090909,358.00000,364.0
2,3,393.0,209.0000,138.000000,146.000000,116.000000,102.0,61.6,97.518519,122.0000,141.090909,410.00000,337.0
3,4,346.0,266.0000,112.000000,90.000000,96.000000,150.0,61.6,87.000000,122.0000,141.090909,405.00000,331.0
4,5,331.0,258.0000,133.000000,129.000000,176.000000,143.0,61.6,70.000000,105.0000,154.000000,300.84375,293.0
5,6,393.0,285.0000,126.000000,63.000000,215.000000,111.0,61.6,81.000000,96.0000,192.000000,385.00000,285.0
6,7,376.0,310.0000,147.000000,143.000000,176.000000,103.5,75.0,97.518519,76.0000,141.090909,300.84375,351.0
7,8,377.0,140.0000,222.000000,157.000000,126.000000,145.0,73.0,97.518519,78.0000,167.000000,300.84375,353.0
8,9,424.0,204.0000,121.000000,245.000000,206.000000,150.0,63.0,117.000000,39.0000,174.000000,448.00000,353.0
9,10,402.0,176.0000,170.000000,204.000000,187.000000,139.0,61.6,97.518519,40.0000,141.090909,305.00000,318.0
